In [ ]:
import os
os.makedirs('/kaggle/working/tools', exist_ok=True)
open('/kaggle/working/tools/__init__.py','w').close()


In [ ]:
%%writefile /kaggle/working/tools/divnet_infer.py
"""Inference for the public ``giorgosi/biohub-divnet-v2`` checkpoint.

The dataset manifest is the contract: four image channels at lags -1, 0, +1,
+2 plus a Gaussian centre marker, XY mean pooling by four, a 16x32x32 crop,
percentile normalisation, and a small 3-D U-Net followed by global pooling.

``read_test_frame`` is supplied by the caller (the public pipeline notebook or
the local B20 scorer). This module raises on an incompatible checkpoint or
tensor shape; a silent geometric fallback would invalidate B20.
"""

from __future__ import annotations

import os
from pathlib import Path
from typing import Callable

import numpy as np
import torch


class _DivNetConvBlock(torch.nn.Module):
    def __init__(self, in_channels: int, out_channels: int) -> None:
        super().__init__()
        self.block = torch.nn.Sequential(
            torch.nn.Conv3d(in_channels, out_channels, 3, padding=1, bias=False),
            torch.nn.GroupNorm(min(8, out_channels), out_channels),
            torch.nn.SiLU(inplace=True),
            torch.nn.Conv3d(out_channels, out_channels, 3, padding=1, bias=False),
            torch.nn.GroupNorm(min(8, out_channels), out_channels),
            torch.nn.SiLU(inplace=True),
        )

    def forward(self, value: torch.Tensor) -> torch.Tensor:
        return self.block(value)


class DivNetMitosisClassifier(torch.nn.Module):
    """Architecture reconstructed exactly from the public checkpoint shapes."""

    def __init__(self, in_channels: int = 5, base_channels: int = 16) -> None:
        super().__init__()
        c = int(base_channels)
        self.enc1 = _DivNetConvBlock(in_channels, c)
        self.pool1 = torch.nn.MaxPool3d(2, 2)
        self.enc2 = _DivNetConvBlock(c, c * 2)
        self.pool2 = torch.nn.MaxPool3d(2, 2)
        self.enc3 = _DivNetConvBlock(c * 2, c * 4)
        self.pool3 = torch.nn.MaxPool3d(2, 2)
        self.bottleneck = _DivNetConvBlock(c * 4, c * 8)
        self.up3 = torch.nn.ConvTranspose3d(c * 8, c * 4, 2, 2)
        self.dec3 = _DivNetConvBlock(c * 8, c * 4)
        self.up2 = torch.nn.ConvTranspose3d(c * 4, c * 2, 2, 2)
        self.dec2 = _DivNetConvBlock(c * 4, c * 2)
        self.up1 = torch.nn.ConvTranspose3d(c * 2, c, 2, 2)
        self.dec1 = _DivNetConvBlock(c * 2, c)
        self.gap = torch.nn.AdaptiveAvgPool3d((1, 1, 1))
        self.head = torch.nn.Linear(c, 1)

    def forward(self, value: torch.Tensor) -> torch.Tensor:
        e1 = self.enc1(value)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        bottleneck = self.bottleneck(self.pool3(e3))
        d3 = self.dec3(torch.cat([self.up3(bottleneck), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.head(self.gap(d1).flatten(1))


def _checkpoint_candidates() -> list[Path]:
    root = Path(__file__).resolve().parent.parent
    explicit = os.environ.get("BIOHUB_DIVNET_CHECKPOINT", "").strip()
    candidates = [Path(explicit)] if explicit else []
    candidates.extend(
        [
            root / "external" / "giorgosi__biohub-divnet-v2" / "best_overall.pt",
            root / "external" / "giorgosi__biohub-divnet-v2" / "weights" / "divnet" / "best_overall.pt",
            Path("/kaggle/input/biohub-divnet-v2/best_overall.pt"),
            Path("/kaggle/input/biohub-divnet-v2/weights/divnet/best_overall.pt"),
            Path("/kaggle/input/datasets/giorgosi/biohub-divnet-v2/best_overall.pt"),
            Path("/kaggle/input/datasets/giorgosi/biohub-divnet-v2/weights/divnet/best_overall.pt"),
        ]
    )
    return candidates


def load_divnet_mitosis_model(checkpoint: Path | None = None) -> dict[str, object]:
    if not globals().get("DIVNET_VERIFY", True):
        raise RuntimeError("DivNet is disabled by configuration")
    path = Path(checkpoint) if checkpoint is not None else next(
        (item for item in _checkpoint_candidates() if item.is_file()), None
    )
    if path is None:
        checked = "\n".join(str(item) for item in _checkpoint_candidates())
        raise FileNotFoundError(f"DivNet checkpoint not found. Checked:\n{checked}")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    payload = torch.load(path, map_location="cpu", weights_only=False)
    state = payload.get("model_state", payload) if isinstance(payload, dict) else payload
    config = payload.get("config", {}) if isinstance(payload, dict) else {}
    model_cfg = config.get("model", {}) if isinstance(config, dict) else {}
    model = DivNetMitosisClassifier(
        in_channels=int(model_cfg.get("in_channels", 5)),
        base_channels=int(model_cfg.get("base_channels", 16)),
    )
    model.load_state_dict(state, strict=True)
    model.to(device).eval()
    return {
        "model": model,
        "device": device,
        "torch": torch,
        "path": path.resolve(),
        "config": config,
        "prepared_frames": {},
    }


def _pool_xy(frame: np.ndarray, factor: int) -> np.ndarray:
    frame = np.asarray(frame)
    if factor <= 1:
        return frame.astype(np.float32, copy=False)
    z, y, x = frame.shape
    y2, x2 = (y // factor) * factor, (x // factor) * factor
    value = frame[:, :y2, :x2].astype(np.float32, copy=False)
    return value.reshape(z, y2 // factor, factor, x2 // factor, factor).mean(axis=(2, 4))


def _normalise(frame: np.ndarray, lo_pct: float, hi_pct: float, clip: tuple[float, float]) -> np.ndarray:
    value = np.asarray(frame, dtype=np.float32)
    lo, hi = np.percentile(value, [lo_pct, hi_pct])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return np.zeros_like(value)
    return np.clip((value - float(lo)) / (float(hi) - float(lo)), clip[0], clip[1]).astype(np.float32)


def _crop_with_padding(frame: np.ndarray, centre: tuple[int, int, int], shape: tuple[int, int, int]) -> np.ndarray:
    result = np.zeros(shape, dtype=np.float32)
    source_slices: list[slice] = []
    target_slices: list[slice] = []
    for coordinate, size, limit in zip(centre, shape, frame.shape):
        start = coordinate - size // 2
        stop = start + size
        source_start, source_stop = max(0, start), min(limit, stop)
        target_start = source_start - start
        target_stop = target_start + max(0, source_stop - source_start)
        source_slices.append(slice(source_start, source_stop))
        target_slices.append(slice(target_start, target_stop))
    result[tuple(target_slices)] = frame[tuple(source_slices)]
    return result


def _marker(shape: tuple[int, int, int], sigma: tuple[float, float, float]) -> np.ndarray:
    axes = [np.arange(size, dtype=np.float32) - size // 2 for size in shape]
    zz, yy, xx = np.meshgrid(*axes, indexing="ij")
    distance = (zz / sigma[0]) ** 2 + (yy / sigma[1]) ** 2 + (xx / sigma[2]) ** 2
    return np.exp(-0.5 * distance).astype(np.float32)


def prepare_divnet_input(
    dataset: str,
    t: int,
    parent_node: dict[str, object],
    divnet_bundle: dict[str, object],
    frame_cache: dict[object, np.ndarray],
    read_frame: Callable[[str, int, dict], np.ndarray] | None = None,
) -> np.ndarray:
    """Build one ``(5, 16, 32, 32)`` sample from the manifest recipe."""
    cfg = divnet_bundle.get("config", {})
    input_cfg = cfg.get("input", {}) if isinstance(cfg, dict) else {}
    norm_cfg = cfg.get("normalization", {}) if isinstance(cfg, dict) else {}
    lags = tuple(int(value) for value in input_cfg.get("image_lags", (-1, 0, 1, 2)))
    pool_xy = int(input_cfg.get("pool_xy", 4))
    shape = (
        int(input_cfg.get("crop_z", 16)),
        int(input_cfg.get("crop_yx", 32)),
        int(input_cfg.get("crop_yx", 32)),
    )
    sigma = tuple(float(value) for value in input_cfg.get("marker_sigma", (1.5, 2.0, 2.0)))
    clip = tuple(float(value) for value in norm_cfg.get("clip", (-0.5, 6.0)))
    reader = read_frame or globals().get("read_test_frame")
    if reader is None:
        raise RuntimeError("DivNet needs a read_frame callback or a global read_test_frame")
    prepared = divnet_bundle.setdefault("prepared_frames", {})
    channels: list[np.ndarray] = []
    for lag in lags:
        frame_t = max(0, int(t) + lag)
        key = (dataset, frame_t)
        if key not in prepared:
            raw = reader(dataset, frame_t, frame_cache)
            pooled = _pool_xy(raw, pool_xy)
            prepared[key] = _normalise(
                pooled,
                float(norm_cfg.get("lo_pct", 50.0)),
                float(norm_cfg.get("hi_pct", 99.5)),
                clip,
            )
            frame_cache.pop(frame_t, None)
        centre = (
            int(round(float(parent_node["z"]))),
            int(round(float(parent_node["y"]) / pool_xy)),
            int(round(float(parent_node["x"]) / pool_xy)),
        )
        channels.append(_crop_with_padding(prepared[key], centre, shape))
    channels.append(_marker(shape, sigma))
    sample = np.stack(channels, axis=0)
    if sample.shape != (5, *shape) or not np.isfinite(sample).all():
        raise RuntimeError(f"Invalid DivNet input: shape={sample.shape}, finite={np.isfinite(sample).all()}")
    return sample


def divnet_score_division(
    dataset: str,
    t: int,
    parent_node: dict[str, object],
    daughter1: dict[str, object],
    daughter2: dict[str, object],
    divnet_bundle: dict[str, object],
    frame_cache: dict[object, np.ndarray],
    read_frame: Callable[[str, int, dict], np.ndarray] | None = None,
) -> float:
    """Return the parent division probability; daughter arguments preserve the public API."""
    del daughter1, daughter2
    sample = prepare_divnet_input(dataset, t, parent_node, divnet_bundle, frame_cache, read_frame)
    torch_mod = divnet_bundle["torch"]
    tensor = torch_mod.from_numpy(sample).unsqueeze(0).to(
        device=divnet_bundle["device"], dtype=torch_mod.float32
    )
    with torch_mod.inference_mode():
        logit = divnet_bundle["model"](tensor)
        probability = float(torch_mod.sigmoid(logit).item())
    if not np.isfinite(probability):
        raise RuntimeError("DivNet produced a non-finite probability")
    return probability


DIVNET_BUNDLE = None


In [ ]:
%%writefile /kaggle/working/tools/daughternet.py
"""DaughterNet: 3-class node classifier (continuation / parent / daughter) for the L2 division stage.

Shared contract between training (Kaggle K-DN kernels), the submission kernel patch (--patch divinsert) and the laptop
replay. The input sample is EXACTLY the DivNet manifest contract of tools/divnet_infer.prepare_divnet_input: image
channels at lags -1, 0, +1, +2 around the node's frame t, XY mean-pool by 4, a 16x32x32 crop centred on the node,
percentile 50/99.5 normalisation clipped to [-0.5, 6], plus one gaussian centre marker -> (5, 16, 32, 32).
The network is the DivNet UNet-GAP body with a 3-logit head. Class order is fixed: CLASSES = (cont, parent, daughter).
  parent   = GT dividing node at the last frame before the split (label at its own t)
  daughter = GT daughter node at its first frame (label at its own t)
  cont     = annotated node with in-degree = out-degree = 1, >= 2 frames from any GT division
Scoring averages softmax over the loaded fold models (ensemble) or uses one fold model per video (OOF).
"""
from __future__ import annotations

from pathlib import Path
from typing import Callable, Sequence

import numpy as np
import torch

try:  # repo layout
    from tools.divnet_infer import _DivNetConvBlock, prepare_divnet_input
except Exception:  # embedded in a notebook cell after divnet_infer's code (names already global)
    _DivNetConvBlock = globals().get("_DivNetConvBlock")
    prepare_divnet_input = globals().get("prepare_divnet_input")

CLASSES = ("cont", "parent", "daughter")
DEFAULT_CFG = {
    "input": {"image_lags": [-1, 0, 1, 2], "pool_xy": 4, "crop_z": 16, "crop_yx": 32, "marker_sigma": [1.5, 2.0, 2.0]},
    "normalization": {"lo_pct": 50.0, "hi_pct": 99.5, "clip": [-0.5, 6.0]},
    "model": {"in_channels": 5, "base_channels": 16, "n_classes": 3},
}


class DaughterNet(torch.nn.Module):
    """DivNet UNet-GAP body (same layer names as DivNetMitosisClassifier) with an n_classes head."""

    def __init__(self, in_channels: int = 5, base_channels: int = 16, n_classes: int = 3) -> None:
        super().__init__()
        c = int(base_channels)
        self.enc1 = _DivNetConvBlock(in_channels, c)
        self.pool1 = torch.nn.MaxPool3d(2, 2)
        self.enc2 = _DivNetConvBlock(c, c * 2)
        self.pool2 = torch.nn.MaxPool3d(2, 2)
        self.enc3 = _DivNetConvBlock(c * 2, c * 4)
        self.pool3 = torch.nn.MaxPool3d(2, 2)
        self.bottleneck = _DivNetConvBlock(c * 4, c * 8)
        self.up3 = torch.nn.ConvTranspose3d(c * 8, c * 4, 2, 2)
        self.dec3 = _DivNetConvBlock(c * 8, c * 4)
        self.up2 = torch.nn.ConvTranspose3d(c * 4, c * 2, 2, 2)
        self.dec2 = _DivNetConvBlock(c * 4, c * 2)
        self.up1 = torch.nn.ConvTranspose3d(c * 2, c, 2, 2)
        self.dec1 = _DivNetConvBlock(c * 2, c)
        self.gap = torch.nn.AdaptiveAvgPool3d((1, 1, 1))
        self.head = torch.nn.Linear(c, int(n_classes))

    def forward(self, value: torch.Tensor) -> torch.Tensor:
        e1 = self.enc1(value)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        b = self.bottleneck(self.pool3(e3))
        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.head(self.gap(d1).flatten(1))


def build_model(cfg: dict | None = None) -> DaughterNet:
    m = (cfg or DEFAULT_CFG)["model"]
    return DaughterNet(int(m["in_channels"]), int(m["base_channels"]), int(m["n_classes"]))


def load_daughternet(paths: Sequence[Path | str], device: str | None = None) -> dict:
    """Load one or more fold checkpoints ({'model_state', 'config'}) -> bundle usable by prepare_sample/score_nodes."""
    dev = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
    models, cfg = [], None
    for p in paths:
        payload = torch.load(Path(p), map_location="cpu", weights_only=False)
        cfg = payload.get("config", DEFAULT_CFG) if isinstance(payload, dict) else DEFAULT_CFG
        model = build_model(cfg)
        model.load_state_dict(payload["model_state"] if "model_state" in payload else payload, strict=True)
        models.append(model.to(dev).eval())
    if not models:
        raise FileNotFoundError("no DaughterNet checkpoints given")
    return {"models": models, "device": dev, "torch": torch, "config": cfg, "prepared_frames": {}}


def prepare_sample(dataset: str, t: int, node: dict, bundle: dict, frame_cache: dict,
                   read_frame: Callable[[str, int, dict], np.ndarray] | None = None) -> np.ndarray:
    """(5, 16, 32, 32) float32 sample, identical to the DivNet contract (bundle['config'] carries input/normalization)."""
    return prepare_divnet_input(dataset, int(t), node, bundle, frame_cache, read_frame)


@torch.no_grad()
def score_nodes(dataset: str, nodes: Sequence[dict], bundle: dict, frame_cache: dict,
                read_frame: Callable[[str, int, dict], np.ndarray] | None = None, batch: int = 64,
                model_index: int | None = None) -> np.ndarray:
    """Softmax probabilities (N, 3) in CLASSES order; mean over fold models unless model_index selects one (OOF)."""
    if not nodes:
        return np.zeros((0, 3), dtype=np.float32)
    order = sorted(range(len(nodes)), key=lambda i: int(nodes[i]["t"]))
    out = np.zeros((len(nodes), 3), dtype=np.float32)
    models = bundle["models"] if model_index is None else [bundle["models"][model_index]]
    dev = bundle["device"]
    for s in range(0, len(order), batch):
        idx = order[s:s + batch]
        x = np.stack([prepare_sample(dataset, int(nodes[i]["t"]), nodes[i], bundle, frame_cache, read_frame) for i in idx])
        xt = torch.from_numpy(x).to(dev, dtype=torch.float32)
        p = torch.stack([torch.softmax(m(xt), dim=1) for m in models]).mean(0)
        out[idx] = p.float().cpu().numpy()
    return out


In [ ]:
%%writefile /kaggle/working/tools/train_dorado.py
"""Train DaughterNet from scratch on the dorado pseudo-labelled shards (plan line L3); no Kaggle label is used.

Input: shards written by tools/dorado_crops.py extract (data_external/dorado/shards/b<block>_z<k>.npz): x (N,5,16,32,32)
float16 in the DivNet / DaughterNet contract (4 image lags + centre marker, real surroundings), y class index in
tools.daughternet.CLASSES order (cont, parent, daughter), zyx node voxel position, win_origin pooled origin of the
emulated 64x64x64 (pooled) Kaggle clip whose percentiles normalised the crop, block, hard, track, partner, win_ok.
Output: {'model_state', 'config'} loadable by tools.daughternet.load_daughternet (strict), + <out>.history.json.

Recipe: hold out --val-blocks (default 417, the last developmental block) for early stopping; AdamW lr 1e-3 wd 0.01,
3% linear warmup then cosine to 0 over --epochs (<= 40), batch 128, class-balanced sampler (each draw picks a class
uniformly, then a sample of it with replacement; epoch = number of training samples), label smoothing 0.1 (the
pseudo-labels are noisy), per-sample augmentation in this order:
  1. window padding: the image channels outside the sample's emulated Kaggle clip are zeroed (what
     divnet_infer._crop_with_padding gives for a node near a real clip border); bit-identical to
     tools.dorado_crops.apply_window_pad (re-implemented here as a mask from the window offset so the three Colab
     files are self-contained; the equality is tested locally). Applied per sample with probability --pad-prob.
     CHOICE (default --pad-prob auto): measured on the 17,143 Kaggle GT evaluation nodes (eval_daughternet.py build,
     all 199 videos) 60.6% of the contract crops touch a clip border (6bba 70%, 44b6 40%; mean padded fraction 0.314
     given padding); the emulated dorado windows touch the crop for 74.3% (b083_z1) / 77.4% (b083_z2) of the samples
     (0.328 given padding). Padding always would over-pad (~75% vs 61%), and p = 0.74 would under-pad (~0.56); auto
     sets p = min(1, 0.606 / touched fraction of the training split) (~0.8), so the padded fraction matches Kaggle
     while the amount per padded sample already matches. A number (e.g. 1 = always) overrides it. The validation
     set gets a fixed seeded mask with the same probability (deterministic across epochs).
  2. flips on z / y / x and a k*90 degree rotation in yx, applied identically to all 5 channels (marker included);
  3. intensity gain U(0.8, 1.2) on the 4 image channels, clipped back to the contract range [-0.5, 6].
Samples whose window failed the Kaggle-clip bounds (win_ok False) are dropped unless --keep-bad-windows.
KAGGLE 44b6 OVERLAP: the Kaggle 44b6 videos are space-time crops of the dorado recording itself (tools/
register_44b6_dorado.py places them -> data_external/dorado/kaggle44b6_in_dorado.json); 6bba is another embryo.
Default (--exclude-kaggle none): all dorado samples are used and config.train.kaggle44b6_leak_free is False; the
evaluator (tools/eval_daughternet.py eval) then reports '44b6_clean' / 'all_clean' blocks without the 44b6 rows whose
crop + lag frames meet a sample of this model's shards (all 8 shards: 304 of the 4,780 placed 44b6 rows, all cont;
21/26 parents and 42/52 daughters are clean, the rest sit in the 8 clips without a reliable placement), and 6bba in
full. Strict mode (--exclude-kaggle auto | PATH): every dorado sample whose footprint (lag frames t-1..t+2, 16x32x32
pooled crop) meets a placed 44b6 clip box (frames [o, o+99], 64x256x256 voxels, + a small margin) is dropped from train
AND val (39% of the 8 shards, 380 of 1,021 parents); config.train.kaggle44b6_leak_free is True only when every 44b6
clip has a reliable placement (63/71 now).
Validation metric (early stopping, best checkpoint): macro cross-entropy without smoothing (mean over classes of the
per-class mean CE) on the held-out blocks; per-class one-vs-rest AUC and hard-negative AUCs are printed each epoch.
If no shard of the validation blocks is present (smoke tests), a lineage-grouped random --val-frac holdout is used.

    python tools/train_dorado.py --shards data_external/dorado/shards --out data_external/l3/dorado_only.pt
    python -m tools.train_dorado --shards /content/shards --out /content/dorado_only.pt          (Colab, T4)
    python tools/train_dorado.py --shards data_external/dorado/shards/b083_z1.npz --max-per-shard 400 --epochs 1 ...
"""
from __future__ import annotations

import argparse
import copy
import glob
import json
import math
import os
import sys
import time
from pathlib import Path

import numpy as np
import torch

_ROOT = Path(__file__).resolve().parent.parent
if str(_ROOT) not in sys.path:  # `python tools/train_dorado.py` (repo) or /content/tools/train_dorado.py (Colab)
    sys.path.insert(0, str(_ROOT))

from tools.daughternet import CLASSES, DEFAULT_CFG, build_model  # noqa: E402
from tools.divnet_infer import _marker  # noqa: E402

N_IMG = len(DEFAULT_CFG["input"]["image_lags"])                        # 4 image channels
CROP = (int(DEFAULT_CFG["input"]["crop_z"]), int(DEFAULT_CFG["input"]["crop_yx"]), int(DEFAULT_CFG["input"]["crop_yx"]))
POOL = int(DEFAULT_CFG["input"]["pool_xy"])
WIN = (64, 64, 64)                                                     # pooled Kaggle clip (64 x 256/4 x 256/4)
CLIP = tuple(float(v) for v in DEFAULT_CFG["normalization"]["clip"])
KAGGLE_PAD_RATE = 0.606          # Kaggle GT eval nodes whose contract crop touches a clip border (17,143 nodes)
DORADO_PAD_RATE = 0.743          # dorado samples whose emulated window pads the crop (b083_z1)
DEFAULT_PAD_PROB = "auto"        # KAGGLE_PAD_RATE / touched fraction of the training split (~0.8)
LAGS = tuple(int(v) for v in DEFAULT_CFG["input"]["image_lags"])
KAGGLE44_FILE = "kaggle44b6_in_dorado.json"   # tools/register_44b6_dorado.py output
KAGGLE_CLIP_VOX = (64, 256, 256)                # Kaggle clip (z, y, x) in full-resolution voxels
KAGGLE_CLIP_T = 100
KAGGLE_MARGIN_VOX = (2, 8, 8)                   # registration slack (shift is refined to ~0.5 um)
KAGGLE_MARGIN_T = 1


def log(*a) -> None:
    print(time.strftime("%H:%M:%S"), *a, flush=True)


# ------------------------------------------------------------------------------------------------------------- shards
def find_shards(specs: list[str]) -> list[Path]:
    """Each spec: a .npz file, a directory (all b*_z*.npz in it) or a glob pattern. Sorted, de-duplicated."""
    out: list[Path] = []
    for s in specs:
        p = Path(s)
        if p.is_dir():
            out += sorted(p.glob("b*_z*.npz"))
        elif p.is_file():
            out.append(p)
        else:
            out += [Path(q) for q in sorted(glob.glob(s))]
    out = [p for p in dict.fromkeys(q.resolve() for q in out) if not p.name.endswith(".tmp.npz")]
    if not out:
        raise FileNotFoundError(f"no shard found for {specs}")
    seen: dict[str, Path] = {}
    for p in out:  # two copies of one (block, z-chunk) shard would double its weight (or repeat val samples)
        if p.name in seen:
            raise RuntimeError(f"shard {p.name} given twice: {seen[p.name]} and {p}")
        seen[p.name] = p
    return out


def window_offsets(zyx: np.ndarray, win_origin: np.ndarray) -> np.ndarray:
    """(N, 3) int: start of the sample's normalisation window in crop coordinates (dorado_crops.window_pad_mask's
    rel_lo): window origin minus the crop corner, with the same python round-half-even centre rounding."""
    z = np.asarray(zyx, np.float64)
    centre = np.stack([np.round(z[:, 0]), np.round(z[:, 1] / POOL), np.round(z[:, 2] / POOL)], axis=1).astype(np.int64)
    return np.asarray(win_origin, np.int64) - (centre - np.array(CROP) // 2)


def find_kaggle_exclusion(spec: str, shard_specs: list[str], paths: list[Path]) -> Path | None:
    """'none' -> None; a path -> that file; 'auto' -> KAGGLE44_FILE in a shard directory or its parent (as given and
    resolved), else None."""
    if spec == "none":
        return None
    if spec != "auto":
        p = Path(spec)
        if not p.is_file():
            raise FileNotFoundError(f"--exclude-kaggle {spec}: no such file")
        return p
    dirs: list[Path] = []
    for s in shard_specs:
        d = Path(s) if Path(s).is_dir() else Path(s).parent
        dirs += [d, d.parent]
    for p in paths:
        dirs += [p.parent, p.parent.parent]
    return next((d / KAGGLE44_FILE for d in dict.fromkeys(dirs) if (d / KAGGLE44_FILE).is_file()), None)


def load_kaggle_clips(path: Path) -> tuple[list[dict], list[str]]:
    """44b6 clips to exclude (every PLACED clip, reliable or not: a wrong placement only drops extra dorado data) and
    the names of the clips without a reliable placement (their 44b6 rows cannot be called leak-free)."""
    d = json.loads(Path(path).read_text())
    clips = [{"stem": s, "o": int(c["o"]), "shift_vox": [float(v) for v in c["shift_vox"]]}
             for s, c in sorted(d["clips"].items()) if c.get("placed")]
    bad = sorted(s for s, c in d["clips"].items() if not c.get("reliable"))
    return clips, bad


def kaggle_overlap(t: np.ndarray, zyx: np.ndarray, clips: list[dict]) -> np.ndarray:
    """(N,) bool: the sample's image footprint (frames t+min(lag) .. t+max(lag), its pooled 16x32x32 crop in
    full-resolution voxels) intersects a registered Kaggle clip (frames [o, o+99], voxels [shift, shift + clip)) grown
    by the registration margin."""
    t = np.asarray(t, np.int64)
    z = np.asarray(zyx, np.float64).reshape(-1, 3)
    hit = np.zeros(len(t), bool)
    if not clips or not len(t):
        return hit
    centre = np.stack([np.round(z[:, 0]), np.round(z[:, 1] / POOL), np.round(z[:, 2] / POOL)], axis=1)
    unit = np.array([1, POOL, POOL])
    lo = (centre - np.array(CROP) // 2) * unit
    hi = lo + np.array(CROP) * unit
    t_lo, t_hi = t + min(LAGS), t + max(LAGS)
    margin = np.array(KAGGLE_MARGIN_VOX)
    for c in clips:
        o, sh = c["o"], np.array(c["shift_vox"])
        tin = (t_hi >= o - KAGGLE_MARGIN_T) & (t_lo <= o + KAGGLE_CLIP_T - 1 + KAGGLE_MARGIN_T)
        sin = np.all((hi > sh - margin) & (lo < sh + np.array(KAGGLE_CLIP_VOX) + margin), axis=1)
        hit |= tin & sin
    return hit


def pad_mask(rel_lo: torch.Tensor) -> torch.Tensor:
    """(B, 3) window offsets -> (B, 1, 16, 32, 32) bool, True = inside the emulated clip (kept)."""
    dev = rel_lo.device
    parts = []
    for ax, (size, w) in enumerate(zip(CROP, WIN)):
        i = torch.arange(size, device=dev)[None, :]
        lo = rel_lo[:, ax:ax + 1]
        parts.append((i >= lo) & (i < lo + w))
    return (parts[0][:, None, :, None, None] & parts[1][:, None, None, :, None] & parts[2][:, None, None, None, :])


def load_shards(paths: list[Path], max_per_shard: int = 0, keep_bad_windows: bool = False, seed: int = 0,
                kaggle_clips: list[dict] | None = None) -> dict:
    """Concatenate shards: x (N, 4, 16, 32, 32) float16 image channels (the marker is re-added at batch time), labels
    and the metadata needed for window padding, hold-out and readout."""
    mk = _marker(CROP, tuple(float(v) for v in DEFAULT_CFG["input"]["marker_sigma"]))
    rng = np.random.default_rng(seed)
    cols: dict[str, list] = {k: [] for k in ("x", "y", "rel_lo", "block", "zchunk", "hard", "group", "shard")}
    infos = []
    for si, p in enumerate(paths):
        t0 = time.time()
        with np.load(p) as d:
            info = json.loads(str(d["info"]))
            if tuple(info.get("classes", CLASSES)) != tuple(CLASSES):
                raise RuntimeError(f"{p.name}: class order {info.get('classes')} != {CLASSES}")
            if (tuple(info["crop"]) != CROP or tuple(info["window"]) != WIN or int(info["pool"]) != POOL
                    or len(info["lags"]) != N_IMG or tuple(info["clip"]) != CLIP):
                raise RuntimeError(f"{p.name}: geometry {info['crop']} {info['window']} {info['pool']} {info['lags']} "
                                   f"does not match the DaughterNet contract")
            y = d["y"].astype(np.int64)
            n = len(y)
            keep = np.ones(n, bool) if keep_bad_windows or "win_ok" not in d.files else d["win_ok"].astype(bool)
            n_bad_window = int(n - keep.sum())
            kag = kaggle_overlap(d["t"], d["zyx"], kaggle_clips or []) & keep
            keep &= ~kag
            idx = np.flatnonzero(keep)
            if max_per_shard and len(idx) > max_per_shard:  # stratified subset (smoke tests)
                sel = []
                for c in range(len(CLASSES)):
                    ic = idx[y[idx] == c]
                    k = int(round(max_per_shard * len(ic) / len(idx)))
                    sel.append(rng.choice(ic, min(len(ic), max(k, 1 if len(ic) else 0)), replace=False))
                idx = np.sort(np.concatenate(sel))
            x = d["x"]
            if x.shape[1:] != (N_IMG + 1, *CROP):
                raise RuntimeError(f"{p.name}: x shape {x.shape}")
            if n and float(np.abs(x[0, N_IMG].astype(np.float32) - mk).max()) > 1e-3:
                raise RuntimeError(f"{p.name}: channel {N_IMG} is not the contract centre marker")
            cols["x"].append(np.ascontiguousarray(x[idx, :N_IMG]).astype(np.float16, copy=False))
            del x
            cols["y"].append(y[idx])
            cols["rel_lo"].append(window_offsets(d["zyx"][idx], d["win_origin"][idx]))
            cols["block"].append(np.full(len(idx), int(info["block"]), np.int64))
            cols["zchunk"].append(np.full(len(idx), int(info["zchunk"]), np.int64))
            cols["hard"].append(d["hard"][idx].astype(bool))
            # lineage group for the random fallback hold-out: a daughter joins its parent's track
            track, partner = d["track"][idx].astype(np.int64), d["partner"][idx].astype(np.int64)
            grp = np.where(y[idx] == CLASSES.index("daughter"), partner, track)
            cols["group"].append(grp + (int(info["block"]) << 32))
            cols["shard"].append(np.full(len(idx), si, np.int64))
            info["n_used"] = int(len(idx))
            info["n_dropped_bad_window"] = n_bad_window
            info["n_dropped_kaggle44b6"] = int(kag.sum())
            info["dropped_kaggle44b6_counts"] = np.bincount(y[kag], minlength=len(CLASSES)).tolist()
            info["path"] = str(p)
            if info.get("window_pad"):
                log(f"WARNING {p.name}: window padding is baked in (--window-pad); --pad-prob has no effect on it")
            infos.append(info)
        log(f"shard {p.name}: {len(idx)}/{n} samples {np.bincount(cols['y'][-1], minlength=3).tolist()} "
            f"(dropped: bad windows {infos[-1]['n_dropped_bad_window']}, inside Kaggle 44b6 clips "
            f"{infos[-1]['n_dropped_kaggle44b6']} {infos[-1]['dropped_kaggle44b6_counts']}) in {time.time() - t0:.1f}s")
    data = {k: np.concatenate(v) for k, v in cols.items()}
    data["infos"] = infos
    return data


def split_train_val(data: dict, val_blocks: list[int], val_frac: float,
                    seed: int) -> tuple[np.ndarray, np.ndarray, str]:
    blocks = data["block"]
    val = np.isin(blocks, val_blocks)
    if val.any() and (~val).any():
        return np.flatnonzero(~val), np.flatnonzero(val), f"blocks {sorted(set(blocks[val].tolist()))}"
    # fallback: lineage-grouped random hold-out (parent and its daughters together)
    groups = np.unique(data["group"])
    rng = np.random.default_rng(seed + 17)
    held = set(rng.choice(groups, max(1, int(round(val_frac * len(groups)))), replace=False).tolist())
    val = np.array([g in held for g in data["group"]])
    return (np.flatnonzero(~val), np.flatnonzero(val),
            f"random lineage-grouped {val_frac:.0%} (no shard of blocks {val_blocks} present)")


# ------------------------------------------------------------------------------------------------------------ metrics
def auc_score(scores, positive) -> float | None:
    s = np.asarray(scores, np.float64)
    y = np.asarray(positive, bool)
    n1, n0 = int(y.sum()), int((~y).sum())
    if n1 == 0 or n0 == 0:
        return None
    order = np.argsort(s, kind="mergesort")
    _, inv, cnt = np.unique(s[order], return_inverse=True, return_counts=True)
    avg = np.cumsum(cnt) - (cnt - 1) / 2.0                             # 1-based average rank of each tie group
    ranks = np.empty(len(s))
    ranks[order] = avg[inv]
    return float((ranks[y].sum() - n1 * (n1 + 1) / 2.0) / (n1 * n0))


def val_metrics(probs: np.ndarray, y: np.ndarray, hard: np.ndarray) -> dict:
    p_true = np.clip(probs[np.arange(len(y)), y], 1e-7, 1.0)
    ce = -np.log(p_true)
    present = [c for c in range(len(CLASSES)) if (y == c).any()]
    nan = float("nan")
    res = {"n": int(len(y)), "macro_ce": float(np.mean([ce[y == c].mean() for c in present])) if present else nan,
           "ce": float(ce.mean()) if len(y) else nan,
           "acc": float((probs.argmax(1) == y).mean()) if len(y) else None}
    for c, name in enumerate(CLASSES):
        res[f"n_{name}"] = int((y == c).sum())
        res[f"auc_{name}"] = auc_score(probs[:, c], y == c)
    hn = hard & (y == 0)
    for c, name in ((1, "parent"), (2, "daughter")):
        m = (y == c) | hn
        res[f"auc_{name}_vs_hard"] = auc_score(probs[m, c], y[m] == c)
    return res


# ------------------------------------------------------------------------------------------------------------- model
def augment(xb: torch.Tensor, rel_lo: torch.Tensor, pad_on: torch.Tensor, marker: torch.Tensor,
            rng: np.random.Generator, gain: tuple[float, float]) -> torch.Tensor:
    """(B, 4, 16, 32, 32) image crops -> (B, 5, 16, 32, 32) float32: window padding (where pad_on), marker, flips
    z/y/x, yx rot90 (all 5 channels identically), gain on the image channels, clip to the contract range."""
    b = xb.shape[0]
    dev = xb.device
    x = xb.float()
    if bool(pad_on.any()):
        keep = pad_mask(rel_lo) | ~pad_on.view(b, 1, 1, 1, 1)
        x = x * keep
    x = torch.cat([x, marker.expand(b, 1, *CROP)], dim=1)
    flips = torch.from_numpy(rng.random((b, 3)) < 0.5).to(dev)
    for axis, dim in enumerate((2, 3, 4)):
        x = torch.where(flips[:, axis].view(b, 1, 1, 1, 1), x.flip(dim), x)
    rot = rng.integers(0, 4, size=b)
    for k in (1, 2, 3):
        sel = np.flatnonzero(rot == k)
        if len(sel):
            idx = torch.from_numpy(sel).to(dev)
            x[idx] = torch.rot90(x[idx], k, dims=(3, 4))
    g = torch.from_numpy(rng.uniform(gain[0], gain[1], size=(b, 1, 1, 1, 1)).astype(np.float32)).to(dev)
    x[:, :N_IMG] = (x[:, :N_IMG] * g).clamp(CLIP[0], CLIP[1])
    return x


@torch.no_grad()
def predict(model, x_img: torch.Tensor, rel_lo: torch.Tensor, pad_on: torch.Tensor, marker: torch.Tensor,
            batch: int, amp: bool) -> np.ndarray:
    model.eval()
    out = []
    for s in range(0, x_img.shape[0], batch):
        xb = x_img[s:s + batch].float()
        po = pad_on[s:s + batch]
        if bool(po.any()):
            xb = xb * (pad_mask(rel_lo[s:s + batch]) | ~po.view(-1, 1, 1, 1, 1))
        x = torch.cat([xb, marker.expand(xb.shape[0], 1, *CROP)], dim=1)
        with torch.autocast(device_type=x.device.type, dtype=torch.float16, enabled=amp):
            logits = model(x)
        out.append(torch.softmax(logits.float(), dim=1).cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0, len(CLASSES)), np.float32)


def _atomic_save(obj, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)


def train(args) -> dict:
    t_start = time.time()
    device = torch.device(args.device if args.device != "auto" else ("cuda" if torch.cuda.is_available() else "cpu"))
    use_cuda = device.type == "cuda"
    amp = use_cuda and not args.no_amp
    if use_cuda:
        torch.backends.cudnn.benchmark = True
    if args.threads:
        torch.set_num_threads(args.threads)
    torch.manual_seed(args.seed)
    rng = np.random.default_rng(args.seed)
    gpu = f" ({torch.cuda.get_device_name(0)})" if use_cuda else ""
    log(f"device {device}{gpu} amp={amp} torch {torch.__version__}")

    paths = find_shards(args.shards)
    kag_path = find_kaggle_exclusion(args.exclude_kaggle, args.shards, paths)
    kag_clips, kag_bad = load_kaggle_clips(kag_path) if kag_path else ([], [])
    leak_free = bool(kag_path) and not kag_bad
    if kag_path is None:
        if args.exclude_kaggle == "auto":
            log(f"WARNING: --exclude-kaggle auto found no {KAGGLE44_FILE}: nothing excluded")
        log("note: Kaggle 44b6 videos are crops of this recording; without --exclude-kaggle the 44b6 metrics must be read "
            "on the evaluator's '44b6_clean' rows (6bba is independent)")
    else:
        log(f"Kaggle 44b6 exclusion: {kag_path} ({len(kag_clips)} registered clips"
            + (f"; no reliable placement: {kag_bad} -> 44b6 only partly leak-free" if kag_bad else "") + ")")
    data = load_shards(paths, args.max_per_shard, args.keep_bad_windows, args.seed, kag_clips)
    tr, va, val_desc = split_train_val(data, args.val_blocks, args.val_frac, args.seed)
    y = data["y"]
    log(f"{len(paths)} shards, {len(y)} samples {np.bincount(y, minlength=3).tolist()} (cont/parent/daughter); "
        f"train {len(tr)} {np.bincount(y[tr], minlength=3).tolist()} | val {len(va)} "
        f"{np.bincount(y[va], minlength=3).tolist()} ({val_desc})")
    if len(tr) == 0 or len(va) == 0:
        raise RuntimeError("empty train or validation split")
    store = torch.float16
    t0 = time.time()
    x_tr = torch.from_numpy(data["x"][tr]).to(device=device, dtype=store)
    x_va = torch.from_numpy(data["x"][va]).to(device=device, dtype=store)
    data["x"] = None                                                   # free the host copy
    rel_tr = torch.from_numpy(data["rel_lo"][tr]).to(device)
    rel_va = torch.from_numpy(data["rel_lo"][va]).to(device)
    y_tr = torch.from_numpy(y[tr]).to(device)
    y_va = y[va]
    hard_va = data["hard"][va]
    log(f"tensors on {device} in {time.time() - t0:.1f}s: train {tuple(x_tr.shape)} {x_tr.dtype}, "
        f"val {tuple(x_va.shape)}")
    # padding: fraction of training samples whose crop pokes out of its window -> pad probability
    rel = data["rel_lo"][tr]
    touched = float(((rel > 0) | (rel + np.array(WIN) < np.array(CROP))).any(1).mean())
    pad_mode = "auto" if str(args.pad_prob) == "auto" else "fixed"
    pad_prob = min(1.0, KAGGLE_PAD_RATE / max(touched, 1e-9)) if pad_mode == "auto" else float(args.pad_prob)
    log(f"window padding: {touched:.3f} of the training samples touch their window border (Kaggle GT nodes "
        f"{KAGGLE_PAD_RATE}); pad prob {pad_prob:.3f} ({pad_mode}) -> expected padded fraction {pad_prob * touched:.3f}")
    val_pad_on = torch.from_numpy(np.random.default_rng(args.seed + 99).random(len(va)) < pad_prob).to(device)
    marker = torch.from_numpy(_marker(CROP, tuple(float(v) for v in DEFAULT_CFG["input"]["marker_sigma"]))).view(
        1, 1, *CROP).to(device)

    cfg = copy.deepcopy(DEFAULT_CFG)
    cfg["classes"] = list(CLASSES)
    cfg["train"] = {"source": "dorado", "script": "tools/train_dorado.py", "shards": [p.name for p in paths],
                    "val": val_desc, "val_blocks": list(args.val_blocks), "epochs": args.epochs, "batch": args.batch,
                    "lr": args.lr, "weight_decay": args.weight_decay, "warmup_frac": args.warmup_frac,
                    "label_smoothing": args.label_smoothing, "pad_prob": pad_prob, "pad_prob_mode": pad_mode,
                    "train_touched_fraction": touched, "kaggle_pad_rate": KAGGLE_PAD_RATE, "gain": list(args.gain),
                    "patience": args.patience, "seed": args.seed, "epoch_samples": args.epoch_samples,
                    "max_per_shard": args.max_per_shard, "keep_bad_windows": args.keep_bad_windows,
                    "n_train": int(len(tr)), "n_val": int(len(va)),
                    "train_counts": np.bincount(y[tr], minlength=3).tolist(),
                    "val_counts": np.bincount(y_va, minlength=3).tolist(), "kaggle_labels_used": False,
                    "kaggle44b6_exclusion": None if kag_path is None else {
                        "file": kag_path.name, "clips": len(kag_clips), "clips_unreliable": kag_bad,
                        "dropped": int(sum(i["n_dropped_kaggle44b6"] for i in data["infos"])),
                        "dropped_counts": np.sum([i["dropped_kaggle44b6_counts"] for i in data["infos"]], 0).tolist()},
                    "kaggle44b6_leak_free": leak_free}
    model = build_model(cfg).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    y_tr_np = y[tr]
    classes_present = [c for c in range(len(CLASSES)) if (y_tr_np == c).any()]
    pools = {c: np.flatnonzero(y_tr_np == c) for c in classes_present}
    epoch_samples = int(args.epoch_samples) or len(tr)
    steps_per_epoch = max(1, math.ceil(epoch_samples / args.batch))
    total_steps = steps_per_epoch * int(args.epochs)
    warmup = max(1, int(args.warmup_frac * total_steps))

    def lr_at(step: int) -> float:
        if step < warmup:
            return args.lr * (step + 1) / warmup
        prog = (step - warmup) / max(1, total_steps - warmup)
        return args.lr * 0.5 * (1.0 + math.cos(math.pi * min(1.0, prog)))

    try:
        scaler = torch.amp.GradScaler("cuda", enabled=amp)
    except Exception:  # older torch
        scaler = torch.cuda.amp.GradScaler(enabled=amp)
    out = Path(args.out)
    hist_path = out.with_name(out.stem + ".history.json")
    best, best_epoch, bad, step = float("inf"), -1, 0, 0
    history = []
    for epoch in range(int(args.epochs)):
        te = time.time()
        model.train()
        cls = rng.choice(classes_present, size=epoch_samples)
        order = np.empty(epoch_samples, np.int64)
        for c in classes_present:
            sel = np.flatnonzero(cls == c)
            order[sel] = rng.choice(pools[c], size=len(sel), replace=True)
        rng.shuffle(order)
        run_loss, seen = torch.zeros((), device=device), 0
        for s in range(0, epoch_samples, args.batch):
            idx = torch.from_numpy(order[s:s + args.batch]).to(device)
            b = len(idx)
            pad_on = torch.from_numpy(rng.random(b) < pad_prob).to(device)
            xb = augment(x_tr[idx], rel_tr[idx], pad_on, marker, rng, tuple(args.gain))
            yb = y_tr[idx]
            for g in opt.param_groups:
                g["lr"] = lr_at(step)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
                logits = model(xb)
            loss = torch.nn.functional.cross_entropy(logits.float(), yb, label_smoothing=args.label_smoothing)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            run_loss += loss.detach() * b
            seen += b
            step += 1
        if use_cuda:
            torch.cuda.synchronize()
        train_loss = float(run_loss) / max(1, seen)
        train_secs = time.time() - te
        probs = predict(model, x_va, rel_va, val_pad_on, marker, args.eval_batch, amp)
        vm = val_metrics(probs, y_va, hard_va)
        val_loss = vm["macro_ce"]
        improved = bool(np.isfinite(val_loss) and val_loss < best)
        if improved:
            best, best_epoch, bad = val_loss, epoch, 0
            cfg["train"].update({"best_epoch": epoch, "best_val_macro_ce": float(val_loss),
                                 "best_val_metrics": {k: v for k, v in vm.items()}})
            _atomic_save({"model_state": {k: v.detach().cpu() for k, v in model.state_dict().items()},
                          "config": copy.deepcopy(cfg)}, out)
        else:
            bad += 1
        if args.save_last:
            _atomic_save({"model_state": {k: v.detach().cpu() for k, v in model.state_dict().items()},
                          "config": copy.deepcopy(cfg), "epoch": epoch}, out.with_name(out.stem + ".last.pt"))
        rec = {"epoch": epoch, "train_loss_smoothed": train_loss, "val_macro_ce": val_loss, "lr": lr_at(step - 1),
               "secs": round(time.time() - te, 1), "train_samples_per_s": round(seen / max(1e-9, train_secs), 1),
               **{f"val_{k}": v for k, v in vm.items()}}
        history.append(rec)
        hist_path.write_text(json.dumps(history, indent=1))
        f = lambda v: "n/a" if v is None else f"{v:.3f}"  # noqa: E731
        log(f"epoch {epoch:2d}: train {train_loss:.4f} | val macro CE {val_loss:.4f}{' *' if improved else '  '} "
            f"AUC cont {f(vm['auc_cont'])} parent {f(vm['auc_parent'])} daughter {f(vm['auc_daughter'])} | "
            f"vs hard: parent {f(vm['auc_parent_vs_hard'])} daughter {f(vm['auc_daughter_vs_hard'])} | "
            f"lr {rec['lr']:.2e} {rec['secs']}s ({rec['train_samples_per_s']} samples/s)")
        if bad >= args.patience:
            log(f"early stop at epoch {epoch} (best epoch {best_epoch}, val macro CE {best:.4f})")
            break
    if best_epoch < 0:
        raise RuntimeError("no finite validation loss: no checkpoint written")
    summary = {"out": str(out), "best_epoch": best_epoch, "best_val_macro_ce": best, "epochs_run": len(history),
               "secs": round(time.time() - t_start, 1), "device": str(device),
               "best_val_metrics": cfg["train"]["best_val_metrics"]}
    log(f"best epoch {best_epoch} val macro CE {best:.4f} -> {out}; history {hist_path}; total {summary['secs']}s")
    return summary


def parse_args(argv=None):
    ap = argparse.ArgumentParser(description=__doc__.split("\n")[0],
                                 formatter_class=argparse.RawDescriptionHelpFormatter, epilog=__doc__)
    ap.add_argument("--shards", nargs="+", default=[str(_ROOT / "data_external" / "dorado" / "shards")],
                    help="shard files, directories (b*_z*.npz inside) or glob patterns")
    ap.add_argument("--out", default=str(_ROOT / "data_external" / "l3" / "dorado_only.pt"))
    ap.add_argument("--val-blocks", type=int, nargs="*", default=[417])
    ap.add_argument("--val-frac", type=float, default=0.15, help="fallback hold-out when no val-block shard exists")
    ap.add_argument("--epochs", type=int, default=40)
    ap.add_argument("--batch", type=int, default=128)
    ap.add_argument("--eval-batch", type=int, default=256)
    ap.add_argument("--lr", type=float, default=1e-3)
    ap.add_argument("--weight-decay", type=float, default=0.01)
    ap.add_argument("--warmup-frac", type=float, default=0.03)
    ap.add_argument("--label-smoothing", type=float, default=0.1)
    ap.add_argument("--pad-prob", default=DEFAULT_PAD_PROB,
                    help="probability of the Kaggle clip-border padding per sample: 'auto' (match the Kaggle padded "
                         "fraction 0.606) or a number in [0, 1] (1 = always)")
    ap.add_argument("--gain", type=float, nargs=2, default=[0.8, 1.2])
    ap.add_argument("--patience", type=int, default=10)
    ap.add_argument("--epoch-samples", type=int, default=0, help="draws per epoch (0 = number of training samples)")
    ap.add_argument("--max-per-shard", type=int, default=0, help="stratified subset per shard (smoke tests)")
    ap.add_argument("--keep-bad-windows", action="store_true", help="keep samples with win_ok False")
    ap.add_argument("--exclude-kaggle", default="none",
                    help=f"strict mode: drop dorado samples inside the registered Kaggle 44b6 clips: a path, 'auto' "
                         f"({KAGGLE44_FILE} next to the shards or in their parent dir), or 'none' (default: use all "
                         f"samples; the evaluator reports overlap-free 44b6 rows)")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--device", default="auto", help="auto | cpu | cuda | cuda:N")
    ap.add_argument("--threads", type=int, default=0, help="torch CPU threads (0 = default)")
    ap.add_argument("--no-amp", action="store_true")
    ap.add_argument("--save-last", action="store_true", help="also write <out>.last.pt every epoch")
    a = ap.parse_args(argv)
    if a.pad_prob != "auto":
        try:
            a.pad_prob = float(a.pad_prob)
        except ValueError:
            ap.error("--pad-prob must be 'auto' or a number in [0, 1]")
        if not 0.0 <= a.pad_prob <= 1.0:
            ap.error("--pad-prob must be 'auto' or a number in [0, 1]")
    if a.epochs < 1:
        ap.error("--epochs must be >= 1")
    return a


def main(argv=None) -> int:
    train(parse_args(argv))
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
import subprocess, sys, glob, os
os.chdir('/kaggle/working')
shards = sorted(set(p for p in glob.glob('/kaggle/input/**/b*_z*.npz', recursive=True)))
seen = {}
for p in shards:
    seen.setdefault(os.path.basename(p), p)
shards = sorted(seen.values())
print(len(shards), 'shards'); [print(' ', s) for s in shards]
cmd = [sys.executable, '-m', 'tools.train_dorado', '--out', '/kaggle/working/dorado_only.pt', '--save-last', '--shards'] + shards
print(' '.join(cmd))
r = subprocess.run(cmd)
assert r.returncode == 0, r.returncode
assert os.path.exists('/kaggle/working/dorado_only.pt')
